# SPARK SQL Learning 
##  Spark SQL Read Write Options with multiple Sources 
## more on Read  Options - By Exploring this - You all becom a Data INGESTION Developer 
connecting Various Sources and bricnging the data into Lake environemnt 

- file systems 
- HDFS , Local , Cloud storages 
- csv , json , xml , parquet , ORC , delta ....
- Databases 
- Data warehouses
- No SQL ..., etc 

# Unity Catalog ( Data objects )
## In Databricks UC Data objects organaised in three namespace 

### Catalog - per domain , per project , per environment
### Schema  - Database 
### tables , volumes , models , fucntions , views 

### Non tabular data(files) we are orgainising in Databricks with volumes 


Access the Volume 


/Volumes/catalog/schema/volume/dir/files

/Volumes/catalog/schema/volume/files

dbfs:/Volumes/catalog/schema/volume/dir/files


dbfs:/Volumes/izwe48catalog/we48db/staging/



In [0]:
%sql
create catalog if not exists izwe48catalog;

create schema if not exists izwe48catalog.we48db;

create volume if not exists izwe48catalog.we48db.staging;


In [0]:
%sql

create schema if not exists test;



### list the file and directory from volume using dbfs file system command

dbfs -> Databricks File system 

### abstarction layer on top of your cloud storages 

### virtual layer on top of cloud storages , s3 /gcs/adls 

### posix (directory structure ) - acces like a normal native file system 



In [0]:
%fs ls dbfs:/Volumes/izwe48catalog/we48db/staging/

### create a directory in dbfs volume 

In [0]:
%fs mkdirs  dbfs:/Volumes/izwe48catalog/we48db/staging/csvdata

In [0]:
%fs head dbfs:/Volumes/izwe48catalog/we48db/staging/custs

## Read delmited file from spark create a dataframe 

### spark.read.csv-> dataframe 


when we write spark program , first spark session we havce to create 

### Notebook spark session already available 

`from pyspark.sql import SparkSession`

`spark=SparkSession.builder.getOrCreate()`

### uri for multiple file system 

uniform resource identifier 

hdfs ->   hdfs:/   ( default in cloudera based hadoop)

dbfs -> dbfs:/    ( default in databricks)

aws s3 -> s3a:/ or s3:/


google cloud storage -> gcs:/ 


azure datalake storage -> adls:/

local linux  -> file:/


In [0]:
# required input -> path 
cust_df=spark.read.csv("dbfs:/Volumes/izwe48catalog/we48db/staging/custs")

print(type(cust_df)) # dataframe


cust_df.show()  # action -> similar like collect , trigger the execution , job created 
# show -> default displaying first 20 records 

cust_df.printSchema() # display the structure / schema / columns of the dataframe with datatypes

### spark.read.csv - defaults

- delimiter -> ,
- col names - _c1,_c1...
- datatype - string 
- header - False (first row)

## chnage the columne names , fix the datatype

-- inferSchema - by reading data create the proper schema 

infeschema -> peformance issue when we deal with huge volume of data

             -->  changing the schema based on data ( not follwoing the fix schema )

In [0]:
cust_df=spark.read.csv("dbfs:/Volumes/izwe48catalog/we48db/staging/custs",inferSchema=True).toDF("custid","firstname","lastname","age","profession")
# show only 5 records 
cust_df.show(5,False)
cust_df.printSchema()

# total number of records in DF

cust_df.count()  # action -> trigger the execution


In [0]:
%fs head /Volumes/izwe48catalog/we48db/staging/custs_header

In [0]:

header_file="/Volumes/izwe48catalog/we48db/staging/custs_header"
cust_df=spark.read.csv(header_file,inferSchema=True,header=True)
# cust_df=spark.read.csv("dbfs:/Volumes/izwe48catalog/we48db/staging/custs_header")

# show only 5 records 
cust_df.show(5,False)
cust_df.printSchema()

# total number of records in DF

cust_df.count()  # action -> trigger the execution

diffrent delimiter  ( | )

In [0]:
%fs head /Volumes/izwe48catalog/we48db/staging/emp1.csv

In [0]:

emp_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/emp1.csv",sep="|",inferSchema=True,header=True)

emp_df.show() 

emp_df.printSchema()

emp_df.count()





instead of reading data for generating schema , we can provoide the schema as well

so that we can fix the schema , as well as fixing the performance regarding the data read 


In [0]:
# provoding the schema we have two options 
# easy approach 

# we can provode schema using ddl string

schema_str="custid int ,fname string,lname string,age int , prof string"

cust_df=spark.read.csv("dbfs:/Volumes/izwe48catalog/we48db/staging/custs",schema=schema_str)
cust_df.show(5)
cust_df.printSchema()
cust_df.count() 

print(cust_df.schema)

In [0]:
# provoiding schema using spark functions 
# programmatically 

from pyspark.sql.types import StructType ,StructField,IntegerType,StringType

# IntgerType -> int
# stringType- str
# StructType ->  record / row / tuple 
# StructField -> column -> name , datatype 


# structType[structField(colname,datypoe),structField(colname,datypoe),structField(colname,datypoe),structField(colname,datypoe)]

schema_cust=StructType([
    StructField("custid",IntegerType()),
    StructField("fname",StringType()),
    StructField("lname",StringType()),
    StructField("age",IntegerType()),
    StructField("profession",StringType())
])

cust_df=spark.read.csv("dbfs:/Volumes/izwe48catalog/we48db/staging/custs",schema=schema_cust)
cust_df.show(5) # select * from table limit 5
cust_df.printSchema()  # desc table 
cust_df.count() # select count(1) from table 



In [0]:

# fixed schema 
emp_schema="id int ,name string,age int "
emp_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/emp1.csv",sep="|",schema=emp_schema,header=True)

emp_df.show() 

emp_df.printSchema()

emp_df.count()

what happen if the data is not matching with schema ?

NULL 

Rject strtegy -?

allow and RCA  , ignore , fail 

-- mode 

we have three options with mode 

1. Permissive(defaault)  - allow all data , unmatched records null , do RCA later 

2. dropmalformed  -  ignore the bad records , procced with further 

3. failfast -  fail the whole application 



In [0]:
emp_schema="id int ,name string,age int "
emp_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/emp1.csv",sep="|",schema=emp_schema,header=True,mode="PERMISSIVE")

emp_df.show() 

emp_df.printSchema()

emp_df.count()

In [0]:
# RCA take the corrupted record out 
# send back to the source 

emp_schema="id int ,name string,age int,error_rec string "
emp_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/emp1.csv",sep="|",schema=emp_schema,header=True,mode="PERMISSIVE",columnNameOfCorruptRecord="error_rec")

emp_df.show(10,False) 

emp_df.printSchema()

emp_df.count()

# emp_df.filter(emp_df.error_rec.isNotNull()).show(10,False)

In [0]:
emp_schema="id int ,name string,age int"
emp_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/emp1.csv",sep="|",schema=emp_schema,header=True,mode="dropmalformed")

emp_df.show(10,False) 

emp_df.printSchema()



In [0]:
emp_schema="id int ,name string,age int"
emp_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/emp1.csv",sep="|",schema=emp_schema,header=True,mode="failfast")

emp_df.show(10,False) 

emp_df.printSchema()


In [0]:
# csv , json , xml , excel , orc , parquet  - built in source 

# external source not a built in source by spark  - generic method 


# df=spark.read.format().load()  - any source - generic method , built in , external source

cust_df=spark.read.option("header","true").option("inferSchema","true").format("csv").load("dbfs:/Volumes/izwe48catalog/we48db/staging/custs_header")


# cust_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/custs_header",inferSchema=True,header=True)


cust_df.printSchema()

cust_df.show(5)



### so for what are all the option from csv we explored 

- header
- inferSchema
- sep
- schema 
- mode 
- columnNameOfCorruptRecord


In [0]:
cust_df=spark.read.option("header","true").option("inferSchema",True).format("csv").load("dbfs:/Volumes/izwe48catalog/we48db/staging/custs_header")

cust_df.show(5)

In [0]:
%fs head /Volumes/izwe48catalog/we48db/staging/csvdata/sales_delhi.csv

### Read the data from Directory 

csv can read the data from single file , single dir , multiple file from diff dir , multiple dir 

In [0]:

sales_schema= "sid int,store string,sales int,sale_dt string"
sales_df = spark.read.csv("/Volumes/izwe48catalog/we48db/staging/csvdata/sales*", header=True,schema=sales_schema)

sales_df.show(5)

sales_df.printSchema()

print(f"Total records in DF {sales_df.count()}")

In [0]:
sales_schema= "sid int,store string,sales int,sale_dt string"
sales_df = spark.read.csv(path=["/Volumes/izwe48catalog/we48db/staging/csvdata/sales_chennai.csv","/Volumes/izwe48catalog/we48db/staging/csvdata/sales*","/Volumes/izwe48catalog/we48db/staging/data"], header=True,schema=sales_schema)

sales_df.show(5)

sales_df.printSchema()

print(f"Total records in DF {sales_df.count()}")

In [0]:
%fs ls /Volumes/izwe48catalog/we48db/staging/salesdata

salesdata 
        year -1 
          month - 1
                a.csv
        year-2
            month 
              b.csv
         
        year -3

In [0]:

# recursiveFileLookup  -> read all subdir and files

# pathGlobFilter -> apply the filter globally (folder , sub folder level as well)

sales_schema= "sid int,store string,sales int,sale_dt string"
sales_df = spark.read.csv(path="/Volumes/izwe48catalog/we48db/staging/salesdata", header=True,schema=sales_schema,recursiveFileLookup=True,pathGlobFilter="sales*")

sales_df.show(5)

sales_df.printSchema()

print(f"Total records in DF {sales_df.count()}")

In [0]:

df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/dir1",recursiveFileLookup=True)
df.show()

### Read json data and create dataframe  
### json is the built in source supported by spark

' {"id":100 ,"name":"raja","age":25}
{"id":101 ,"name":"jaya","age":28}
{"id":104 ,"name":"suresh","age":42}
{"id":103 ,"name":"krish","age":22}
`

In [0]:
ddl_str="id int,name string,age int"
emp_df=spark.read.json("/Volumes/izwe48catalog/we48db/staging/emp.json",schema=ddl_str)
emp_df.show() 
emp_df.printSchema()

emp_df.count()


# Read the data from Table (hive / databricks  sql )

In [0]:
cust_df=spark.read.table("izwe48catalog.we48db.tbl_customer")

cust_df.printSchema()

cust_df.show(5)

In [0]:
# sql based read in spark 

sql_cust_df=spark.sql("select * from izwe48catalog.we48db.tbl_customer")
sql_cust_df.show(2)
sql_cust_df.printSchema()

In [0]:
%fs head /Volumes/izwe48catalog/we48db/staging/emp2.csv

In [0]:
emp_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/emp2.csv",header=True,sep=",")
emp_df.show()
emp_df.printSchema()


In [0]:
emp_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/emp2.csv",header=True,sep=",",quote="'",escape="|",comment="#")
emp_df.show()
emp_df.printSchema()


In [0]:
%fs head /Volumes/izwe48catalog/we48db/staging/salesdata/sales_chennai.csv

In [0]:
# spark -> yyyy-MM-dd
sales_schema="sid int,store string,sales int,sale_dt date"
sale_df=spark.read.schema(sales_schema).csv("/Volumes/izwe48catalog/we48db/staging/salesdata/sales_chennai.csv",header=True,sep=",",quote="'",escape="|",comment="#",dateFormat="MM/dd/yyyy")
sale_df.show(5)
sale_df.printSchema()

In [0]:
# read json 

emp_df=spark.read.json("/Volumes/izwe48catalog/we48db/staging/emp.json")

emp_df.show()

emp_df.printSchema()

## JSON Features 

In [0]:
%fs head /Volumes/izwe48catalog/we48db/staging/emp2.json

In [0]:
# multiline json , one record come as sigle line is supported or not -> yes 

# multiline with unquted field , next line its not reading -- check

emp_schema="id long,name string,dob date"
emp_df=spark.read.json("/Volumes/izwe48catalog/we48db/staging/emp2.json",multiLine=True,allowUnquotedFieldNames=True,dateFormat="dd/MM/yyyy",allowSingleQuotes=True)

emp_df.show()

emp_df.printSchema()

In [0]:
emp_schema="id long,name string,dob date"
emp_df=spark.read.json("/Volumes/izwe48catalog/we48db/staging/emp3.json",allowUnquotedFieldNames=True,dateFormat="dd/MM/yyyy",allowSingleQuotes=True)

emp_df.show()

emp_df.printSchema()

In [0]:
%fs head /Volumes/izwe48catalog/we48db/staging/emp2.json

In [0]:
emp_df2=spark.read.json("/Volumes/izwe48catalog/we48db/staging/emp2.json",multiLine=True)
emp_df2.show()


CSV 

XML

JSON

Databricks Table (delta)

python List 


## binarty / serilized File format 

- converting human readable format data in machine format (binary format)

- reduce the size , save space , easy to transfer 

### parquet (columnar file format)

-- columnar / serilized / intelligence / self described  file format 

### ORC  - optimized row columnar format 





In [0]:
cust_df=spark.read.parquet("/Volumes/izwe48catalog/we48db/staging/cust_parq/part-00000-tid-7142247718544569994-d3aeab9d-1060-418e-94c4-72d7d89757c2-177-1.c000.snappy.parquet")

cust_df.show()

cust_df.printSchema()

In [0]:
cust_df=spark.read.orc("/Volumes/izwe48catalog/we48db/staging/cust_orc/")

cust_df.show()

cust_df.printSchema()

## schema evolution  

## schema changes 


day 1 to day 10 -> id ,name,age 

day 11 to day 30 ->  id ,name,age ,city 

day 31 to ->  id ,name,year,city 






In [0]:
csv_emp_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/empdata",header=True)


csv_emp_df.show()

csv_emp_df.printSchema()


In [0]:
csv_emp_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/empdata/day1.csv",header=True,inferSchema=True)
csv_emp_df.write.mode("overwrite").parquet("/Volumes/izwe48catalog/we48db/staging/emp_parq_data")

csv_emp_df1=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/empdata/day2.csv",header=True,inferSchema=True)
csv_emp_df1.write.mode("append").parquet("/Volumes/izwe48catalog/we48db/staging/emp_parq_data")

csv_emp_df2=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/empdata/day3.csv",header=True,inferSchema=True)
csv_emp_df2.write.mode("append").parquet("/Volumes/izwe48catalog/we48db/staging/emp_parq_data")

csv_emp_df3=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/empdata/day4.csv",header=True,inferSchema=True)
csv_emp_df3.write.mode("append").parquet("/Volumes/izwe48catalog/we48db/staging/emp_parq_data")

In [0]:
emp_df=spark.read.parquet("/Volumes/izwe48catalog/we48db/staging/emp_parq_data",mergeSchema=True)

emp_df.show()

emp_df.printSchema()

In [0]:
emp_df=spark.read.parquet("/Volumes/izwe48catalog/we48db/staging/emp_parq_data/part*",mergeSchema=True)